# 🧪 ModelFlow Results Viewer & ERD Visualizations

This notebook provides a comprehensive view of the three distinct modeling methodologies implemented in the workshop:
1. **Data Vault 2.0** (Raw Vault & Business Vault)
2. **Inmon 3rd Normal Form** (CDW)
3. **Star Schema** (Dimensional Modeling)

---

In [8]:
import duckdb
import pandas as pd
from IPython.display import display, Markdown, clear_output, SVG
from pathlib import Path
import os
import ipywidgets as widgets
import graphviz

# ── Connections
DB_PATH = Path('data/modelflow.duckdb')
con = None

def get_connection():
    global con
    if con is None:
        try:
            con = duckdb.connect(str(DB_PATH), read_only=True)
        except Exception as e:
            return None
    return con

def check_db_state():
    c = get_connection()
    if c is None: return "LOCKED (By another process)"
    try:
        c.execute("SELECT 1 FROM main_dv_gold.bv_sales_summary LIMIT 1")
        return "READY (Solution Tables Found)"
    except:
        return "NOT BUILT (Solution Tables Missing)"

def query_to_df(sql):
    c = get_connection()
    if c is None:
         return pd.DataFrame({"Error": ["Database is LOCKED. Use the 'Release DB Lock' button below."]})
    try:
        return c.execute(sql).df()
    except Exception as e:
        mode = os.environ.get('DBT_MODE', 'exercise').upper()
        return pd.DataFrame({"Error": [f"Table not found in {mode} mode. Please ensure dbt has run successfully in solution mode."]})

def render_mermaid(diagram):
    display(Markdown(f"""
```mermaid
{diagram}
```
"""))

def generate_erd_svg(schema_name):
    """
    Generates an SVG string representation of the schema using Graphviz.
    """
    c = get_connection()
    if c is None:
         return None
    tables = [r[0] for r in c.execute(f"SELECT table_name FROM information_schema.tables WHERE table_schema = '{schema_name}'").fetchall()]
    
    dot = graphviz.Digraph(comment=schema_name)
    dot.attr(rankdir='LR', size='10,10', bgcolor='transparent')
    dot.attr('node', shape='none', margin='0', fontname='Helvetica', fontsize='12')
    dot.attr('edge', color='#888888')
    
    for table in tables:
        cols = c.execute(f"DESCRIBE {schema_name}.{table}").fetchall()
        
        # Color assignment heuristice
        tn = table.lower()
        bg_color = "#555555"
        if 'hub' in tn or 'dim' in tn: bg_color = "#4e79a7"
        elif 'link' in tn or 'fact' in tn: bg_color = "#f28e2b"
        elif 'sat' in tn: bg_color = "#59a14f"

        # Build HTML-like label for node
        rows = ""
        for col in cols:
            rows += f'<tr><td align="left">{col[0]}</td><td align="right"><font color="#bbbbbb"><i>{col[1]}</i></font></td></tr>'

        label = f'''<<table border="0" cellborder="1" cellspacing="0" cellpadding="4">
          <tr><td bgcolor="{bg_color}" colspan="2"><font color="white"><b>{table}</b></font></td></tr>
          {rows}
        </table>>'''
        
        dot.node(table, label)

    for table in tables:
        cols = [c[0].lower() for c in c.execute(f"DESCRIBE {schema_name}.{table}").fetchall()]
        for other_table in tables:
            if table != other_table:
                if table.lower().startswith(('link_', 'sat_')) and other_table.lower().startswith('hub_'):
                    pot_key = other_table.lower().replace('hub_', '') + '_pk'
                    if pot_key in cols: dot.edge(table, other_table)
                if table.lower().startswith(('fact_', 'mart_', 'rpt_')) and other_table.lower().startswith('dim_'):
                    s_name = other_table.lower().replace('dim_', '')
                    if any(k in cols for k in [s_name + '_key', s_name + '_id']): dot.edge(table, other_table)
    return dot.pipe(format='svg').decode('utf-8')


# ── Interactive Mode Switcher
display(Markdown("### 🕹️ Workshop Control Panel"))

mode_dropdown = widgets.Dropdown(
    options=['EXERCISE', 'SOLUTION'],
    value=os.environ.get('DBT_MODE', 'exercise').upper(),
    description='DBT Mode:',
    style={'description_width': 'initial'}
)

lock_button = widgets.Button(
    description='🔓 Release DB Lock',
    button_style='warning',
    tooltip='Close database connection so dbt can run'
)

output = widgets.Output()

def on_mode_change(change):
    with output:
        clear_output()
        os.environ['DBT_MODE'] = change['new'].lower()
        status_update()

def on_lock_click(b):
    global con
    with output:
        clear_output()
        if con:
            con.close()
            con = None
            display(Markdown("**✅ Connection Closed.** You can now run `dbt run` in your terminal!"))
            display(Markdown("> *Note: Re-run this cell when dbt is finished to reconnect.* "))
        else:
            display(Markdown("**Connection already closed.**"))

def status_update():
    db_state = check_db_state()
    mode = os.environ.get('DBT_MODE', 'exercise').upper()
    display(Markdown(f"**Active Mode:** `{mode}`"))
    display(Markdown(f"**Database status:** `{db_state}`"))
    if mode == 'SOLUTION' and 'NOT BUILT' in db_state:
         display(Markdown("> ⚠️ **Note**: Run `dbt run` in your terminal to build the solution tables!"))
    if 'NOT BUILT' in db_state or 'LOCKED' in db_state:
         display(Markdown("> 💡 **Tip**: Use the 'Release DB Lock' button to let dbt run safely."))

mode_dropdown.observe(on_mode_change, names='value')
lock_button.on_click(on_lock_click)

display(widgets.HBox([mode_dropdown, lock_button]), output)

# Initial state
with output:
    status_update()
from erd_generator import generate_erd

### 🕹️ Workshop Control Panel

Output()

In [9]:
display(Markdown("### 🗄️ Database Metadata Explorer"))
meta_df = query_to_df("""
    SELECT 
        table_schema, 
        table_name, 
        table_type 
    FROM information_schema.tables 
    WHERE table_schema LIKE 'main_%' 
    ORDER BY table_schema, table_name
""")

if not meta_df.empty and "Error" not in meta_df.columns:
    display(meta_df)
else:
    display(Markdown("> *No tables found yet. Build your models in **SOLUTION** mode to see metadata.* "))

### 🗄️ Database Metadata Explorer

,table_schema,table_name,table_type
0,main_dv_bronze,bronze_dv_stg_customers,VIEW
1,main_dv_bronze,bronze_dv_stg_orders,VIEW
2,main_dv_bronze,bronze_dv_stg_products,VIEW
3,main_dv_gold,bv_customer_latest,BASE TABLE
4,main_dv_gold,bv_sales_summary,BASE TABLE
5,main_dv_silver,hub_customer,BASE TABLE
6,main_dv_silver,hub_product,BASE TABLE
7,main_dv_silver,link_order,BASE TABLE
8,main_dv_silver,sat_customer_details,BASE TABLE
9,main_dv_silver,sat_order_details,BASE TABLE


In [10]:
display(Markdown("### 📐 Interactive Schema Explorer"))

tab_contents = ['Data Vault', 'Inmon 3NF', 'Star Schema']
children = []

for title in tab_contents:
    out = widgets.Output()
    with out:
        if title == 'Data Vault':
            schema = 'main_dv_silver'
            display(Markdown("**Data Vault 2.0** uses Hubs (Business Keys), Links (Relationships), and Satellites (Context) for maximum auditability and scale."))
        elif title == 'Inmon 3NF':
            schema = 'main_inmon_silver'
            display(Markdown("**Inmon 3NF** focuses on data integrity, non-redundancy, and normalization for a governance-heavy Corporate Data Warehouse."))
        else:
            schema = 'main_ss_silver'
            display(Markdown("**Star Schema** is the industry standard for BI, using Dimensions (SCD2) and Fact tables for high-performance analytics."))
        
        try:
            svg_data = generate_erd_svg(schema)
            if svg_data:
                 display(SVG(svg_data))
            else:
                 display(Markdown("> *Make sure database is unlocked and solution tables exist.* "))
        except Exception as e:
            display(Markdown(f"> ⚠️ *Could not generate interactive ERD: {e}* "))
            
    children.append(out)

tab = widgets.Tab()
tab.children = children
for i in range(len(tab_contents)):
    tab.set_title(i, tab_contents[i])

display(tab)

### 📐 Interactive Schema Explorer

---

## Detailed Data Previews

Below you can find the high-level metrics and sample data for each architecture when built in **SOLUTION** mode.

In [11]:
display(Markdown("### 🥇 Data Vault Gold Sample"))
display(query_to_df("SELECT * FROM main_dv_gold.bv_sales_summary LIMIT 5"))

### 🥇 Data Vault Gold Sample

,order_id,hk_customer,hk_product,cust_name,cust_country,cust_segment,product_name,category,price,cost,order_date,quantity,discount_pct,order_status,channel,payment_method,gross_revenue,total_cost
0,12,c5ab0bc60ac7929182aadd08703f1ec6,70efdf2ec9b086079795c442636b55fb,Martin Hall,Georgia,Consumer,Face-to-face asynchronous productivity,Home,181.35,139.25,2025-03-29,3,15.0,Shipped,Web,Bank Transfer,462.44,417.75
1,53,35cf8659cfcb13224cbd47863a34fc58,a5bfc9e07964f8dddeb95fc584cd965d,Krystal Carter,Kyrgyz Republic,Home Office,Switchable zero tolerance process improvement,Home,223.29,152.05,2025-07-02,10,0.0,Shipped,Mobile,PayPal,2232.90,1520.50
2,58,9cfdf10e8fc047a44b08ed031e1f0ed1,4e732ced3463d06de0ca9a15b6153677,Ryan Wiley,Costa Rica,Corporate,Progressive executive open architecture,Sports,192.76,96.06,2025-06-04,10,0.0,Shipped,Mobile,Bank Transfer,1927.60,960.60
3,67,5807a685d1a9ab3b599035bc566ce2b9,b6d767d2f8ed5d21a44b0e5886680cb9,Jacqueline Chen,El Salvador,Corporate,User-centric client-server intranet,Home,205.52,76.88,2025-12-19,4,0.0,Shipped,Web,Credit Card,822.08,307.52
4,97,6da9003b743b65f4c0ccd295cc484e57,c81e728d9d4c2f636f067f89cc14862c,Derek Wilson,Qatar,Corporate,Ergonomic impactful analyzer,Sports,104.39,136.82,2025-03-17,6,0.0,Cancelled,Mobile,PayPal,626.34,820.92


In [14]:
display(Markdown("### 🥈 Inmon 3NF Gold Sample"))
display(query_to_df("SELECT * FROM main_inmon_gold.rpt_sales_by_customer LIMIT 5"))

### 🥈 Inmon 3NF Gold Sample

,cust_id,cust_name,cust_country,cust_segment,cust_city,total_orders,total_units_sold,total_gross_revenue,total_cost,total_gross_margin,avg_order_value,first_order_date,last_order_date,customer_lifetime_days
0,993,Erin Hudson,Namibia,Consumer,Michelleton,4,34.0,8967.55,2677.75,6289.80,2241.89,2025-05-06,2026-01-27,266
1,935,Joseph Peterson,Albania,Home Office,Patriciahaven,5,34.0,8247.66,3764.58,4483.08,1649.53,2025-05-13,2026-02-25,288
2,346,Brett Martin,Bhutan,Corporate,Garnerview,4,33.0,7911.60,4824.43,3087.17,1977.90,2025-05-09,2025-12-01,206
3,534,Debbie Gordon,Russian Federation,Consumer,Thomasville,3,21.0,7664.22,3999.55,3664.67,2554.74,2025-10-20,2026-02-13,116
4,515,Ronald Brown,Oman,Corporate,West Gloria,5,38.0,7312.06,7077.03,235.03,1462.41,2025-06-02,2026-02-03,246


In [13]:
display(Markdown("### 🥉 Star Schema Gold Sample"))
display(query_to_df("SELECT * FROM main_ss_gold.mart_sales_dashboard LIMIT 5"))

### 🥉 Star Schema Gold Sample

,order_id,cust_id,cust_name,cust_email,cust_country,cust_city,cust_segment,prod_id,product_name,category,...,payment_method,order_status,quantity,unit_price,unit_cost,discount_pct,gross_revenue,total_cost,gross_margin,gross_margin_pct
0,1855,957,Robert Cline,nathan62@example.com,China,New Tammy,Corporate,17,Face-to-face asynchronous productivity,Home,...,PayPal,Shipped,9,181.35,139.25,10.0,1468.94,1253.25,215.69,14.68
1,1779,130,Shawn Deleon,katrinaanderson@example.net,United States Minor Outlying Islands,Port Davidberg,Home Office,3,Switchable solution-oriented Graphic Interface,Sports,...,PayPal,Shipped,7,247.66,49.97,0.0,1733.62,349.79,1383.83,79.82
2,1379,135,Haley Ross PhD,floressteven@example.com,Korea,East Colin,Home Office,21,Enterprise-wide attitude-oriented task-force,Electronics,...,Crypto,Shipped,10,435.98,112.35,20.0,3487.84,1123.50,2364.34,67.79
3,1277,457,Linda Love,burtonkimberly@example.com,Italy,New Joeville,Consumer,5,Implemented responsive interface,Apparel,...,Crypto,Shipped,7,47.05,10.98,10.0,296.41,76.86,219.55,74.07
4,745,738,Melanie Oliver,figueroajohn@example.net,Algeria,Port Daniel,Corporate,21,Enterprise-wide attitude-oriented task-force,Electronics,...,Credit Card,Returned,7,435.98,112.35,10.0,2746.67,786.45,1960.22,71.37


---

In [12]:
display(Markdown("### 📊 Architecture Comparison Metrics"))
metrics = query_to_df("""
    SELECT 
        'Data Vault' as Methodology, 
        (SELECT COUNT(*) FROM main_dv_silver.hub_customer) as Customer_Count,
        (SELECT COUNT(*) FROM main_dv_gold.bv_sales_summary) as Sales_Fact_Count
    UNION ALL
    SELECT 
        'Inmon 3NF' as Methodology,
        (SELECT COUNT(*) FROM main_inmon_silver.dim_product_3nf),
        (SELECT COUNT(*) FROM main_inmon_silver.fact_order_3nf)
    UNION ALL
    SELECT 
        'Star Schema' as Methodology,
        (SELECT COUNT(DISTINCT cust_id) FROM main_ss_silver.dim_customer_scd2),
        (SELECT COUNT(*) FROM main_ss_silver.fact_sales)
""")
display(metrics)

### 📊 Architecture Comparison Metrics

,Methodology,Customer_Count,Sales_Fact_Count
0,Data Vault,1000,2000
1,Inmon 3NF,50,2000
2,Star Schema,1000,2000
